# Agent Loop Registry Demo

This notebook shows the new app-compatible research loop API. The backend no longer needs to know the LangGraph loop internals; it calls a registered loop by name using the same function signature every time.

The built-in `grammar` loop lives in `packages.research.agent_loops.grammar_loop`. `packages.pipeline.grammar_graph` remains the grammar catalog and tooling layer, with a compatibility shim for older imports.

In [1]:
from pathlib import Path
import sys


def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "packages" / "research").exists() and (candidate / "packages" / "pipeline").exists():
            return candidate
    raise RuntimeError("Could not locate repo root from notebook cwd.")


repo_root = find_repo_root(Path.cwd().resolve())
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

repo_root

PosixPath('/Users/thorbthorb/Downloads/IL_ideation')

## Contract

Every app-compatible loop is a callable with this shape:

```python
def loop(
    prompt: str,
    initial_state: dict[str, Any] | None = None,
    *,
    config: AgentLoopConfig | None = None,
) -> AgentLoopResult:
    ...
```

`AgentLoopResult.hitl` is the backend-facing payload. Keep that payload compatible with the existing grammar HITL shape if you want the current app UI to work unchanged.

In [2]:
from typing import Any

from packages.research.agent_loops import (
    AgentLoopConfig,
    AgentLoopResult,
    list_agent_loops,
    register_agent_loop,
    run_agent_loop,
)

list_agent_loops()

['grammar']

## Run The Existing Grammar Loop

The existing loop is registered as `grammar`. It still uses LangGraph when available, Langfuse observations, and the same final HITL payload used by `/designs/generate`.

In [3]:
result = run_agent_loop(
    "grammar",
    "make a compact quadruped that can climb rocky terrain",
    {"prompt": "make a compact quadruped that can climb rocky terrain", "candidates": [], "population": 2},
    config=AgentLoopConfig(population=2, max_attempts=1),
)

{
    "compile_safe": result.hitl["compile_safe"],
    "population": result.hitl["population"],
    "rule_count": len(result.hitl["structural_rules"]),
    "messages": result.hitl["messages"],
}

{'compile_safe': True,
 'population': 2,
 'rule_count': 5,
 'messages': ['normalize_query: TaskIntent ready.',
  'make_initial_checklist: evaluator checklist ready.',
  'rule_builder_attempt_1: structural rules generated.',
  'resolve_grammar_node_names: fuzzy node-name resolution complete.',
  'compile_structural_rules: compile_safe=True; invalid_grammar_nodes=[].',
  'evaluate_rules: evaluator pass complete.']}

## Register A New Loop In The Notebook

This toy loop returns the same HITL shape as the grammar loop. In real experiments, replace the body with your LangGraph `StateGraph` construction and `graph.invoke(...)`, then return `AgentLoopResult(state=..., hitl=...)`.

In [ ]:
def tiny_demo_loop(
    prompt: str,
    initial_state: dict[str, Any] | None = None,
    *,
    config: AgentLoopConfig | None = None,
) -> AgentLoopResult:
    loop_config = config or AgentLoopConfig()
    population = loop_config.population or 1
    state = {
        "prompt": prompt,
        "initial_state": initial_state or {},
        "population": population,
        "structural_rules": {"structural_rules": {"S": ["BODY"]}},
        "compile_safe": True,
        "messages": ["tiny_demo_loop: returned deterministic app-compatible payload."],
    }
    hitl = {
        "spec": {"task_goal": prompt},
        "inferred_morphology": "biped",
        "checklist": {"criteria": []},
        "structural_rules": {"S": ["BODY"]},
        "node_resolution": {"corrections": [], "unresolved": []},
        "invalid_grammar_nodes": [],
        "population": population,
        "compile_safe": True,
        "compile_error": None,
        "awaiting_human": False,
        "langfuse_trace_id": None,
        "messages": state["messages"],
    }
    return AgentLoopResult(state=state, hitl=hitl)


register_agent_loop("tiny_demo", tiny_demo_loop)
list_agent_loops()

In [ ]:
demo_result = run_agent_loop(
    "tiny_demo",
    "make a tiny inspection walker",
    {"prompt": "make a tiny inspection walker", "candidates": [], "population": 3},
    config=AgentLoopConfig(population=3),
)

demo_result.hitl

## Exercise The Backend Route In-Process

A runtime-registered notebook loop is visible to this Python process. If you run FastAPI in a separate process, register the loop from importable code before starting the server, then pass `agent_loop` in the request body.

In [ ]:
import json
import tempfile

from apps.api.routes import designs
from apps.api.workspace_store import WorkspaceStore

workspace_path = Path(tempfile.mkdtemp()) / "agent-loop-demo.sqlite3"
designs.workspace_store = WorkspaceStore(workspace_path)
designs.log_prompt_query = lambda **_kwargs: "notebook-demo-event"

designs.workspace_store.save_ingest_job(
    {
        "id": "notebook-job-1",
        "source_url": None,
        "er16_plan_json": json.dumps({"task_goal": "make a tiny inspection walker"}),
        "status": "analysis_ready",
    }
)

response = designs.generate_designs(
    designs.GenerateDesignsRequest(
        ingest_job_id="notebook-job-1",
        population=3,
        agent_loop="tiny_demo",
    )
)

{
    "agent_loop": response["agent_loop"],
    "population": response["population"],
    "candidate_count": len(response["candidates"]),
    "rules": response["grammar_hitl"]["structural_rules"],
}

## Server Request Shape

For a running backend, the request body only needs the loop name:

```python
import requests

requests.post(
    "http://localhost:8000/designs/generate",
    json={
        "ingest_job_id": "existing-ingest-job-id",
        "population": 6,
        "agent_loop": "grammar",
    },
).json()
```

Use `GET /designs/agent-loops` to see loop names available inside that server process.